# Phase 4: Graph-Based Feature Engineering

For each transaction, features capture its network context (shared cards, emails, devices, etc.) to detect fraud patterns beyond tabular signals.

Entities: uid, card1, P_emaildomain, R_emaildomain, addr1, DeviceInfo, id_31
Features:
Degree (shared usage)
Neighbor fraud rate (train-only)
Velocity (rolling transaction counts)
Amount aggregates (mean/std/ratios)

In [ ]:
import numpy as np
import pandas as pd

DATA_IN  = "/Users/shengqu/Desktop/CSCI5253/FinalProject/ieee-fraud-detection/data/train_with_uid.parquet"
DATA_OUT = "/Users/shengqu/Desktop/CSCI5253/FinalProject/ieee-fraud-detection/data/graph_features.parquet"

TRAIN_FRACTION = 0.80
# Small additive smoothing for neighbor fraud rate 
SMOOTHING_ALPHA = 10

## 1. Load & time-based split

In [2]:
df = pd.read_parquet(DATA_IN).sort_values("TransactionDT").reset_index(drop=True)
split_idx = int(len(df) * TRAIN_FRACTION)
df["is_train"] = np.arange(len(df)) < split_idx

train_df = df[df["is_train"]]
print(f"Train rows: {len(train_df):,}  |  Total rows: {len(df):,}")
print(f"Global train fraud rate: {train_df['isFraud'].mean():.4%}")

Train rows: 472,432  |  Total rows: 590,540
Global train fraud rate: 3.5135%


## 2. Degree features

In [ ]:
entities = ["uid", "card1", "P_emaildomain", "R_emaildomain",
            "addr1", "DeviceInfo", "id_31"]

# count distinct `neighbor` per `entity`
pairs = [
    ("uid", "card1"),
    ("uid", "P_emaildomain"),
    ("uid", "DeviceInfo"),
    ("uid", "addr1"),
    ("card1", "uid"),
    ("card1", "P_emaildomain"),
    ("card1", "addr1"),
    ("P_emaildomain", "uid"),
    ("P_emaildomain", "card1"),
    ("addr1", "uid"),
    ("addr1", "card1"),
    ("DeviceInfo", "uid"),
]

deg_features = {}
for e, n in pairs:
    key = f"{e}__nunique_{n}"
    deg_features[key] = df.groupby(e)[n].transform("nunique")

deg_df = pd.DataFrame(deg_features)
print("Degree features created:", list(deg_df.columns))
deg_df.head()

Degree features created: ['uid__nunique_card1', 'uid__nunique_P_emaildomain', 'uid__nunique_DeviceInfo', 'uid__nunique_addr1', 'card1__nunique_uid', 'card1__nunique_P_emaildomain', 'card1__nunique_addr1', 'P_emaildomain__nunique_uid', 'P_emaildomain__nunique_card1', 'addr1__nunique_uid', 'addr1__nunique_card1', 'DeviceInfo__nunique_uid']


,uid__nunique_card1,uid__nunique_P_emaildomain,uid__nunique_DeviceInfo,uid__nunique_addr1,card1__nunique_uid,card1__nunique_P_emaildomain,card1__nunique_addr1,P_emaildomain__nunique_uid,P_emaildomain__nunique_card1,addr1__nunique_uid,addr1__nunique_card1,DeviceInfo__nunique_uid
0,1,0,0,1,26,12,11,NaN,NaN,10979.0,1332.0,NaN
1,1,1,0,1,397,26,43,101171.0,8933.0,18443.0,1645.0,NaN
2,1,1,0,1,555,15,44,2737.0,898.0,11407.0,1410.0,NaN
3,1,1,0,1,1694,27,34,47016.0,5751.0,4541.0,748.0,NaN
4,1,1,1,1,8,5,7,101171.0,8933.0,2050.0,439.0,7.0


## 3. Neighbor fraud rate

In [ ]:
global_rate = train_df["isFraud"].mean()

def smoothed_fraud_rate(train_slice, entity_col, alpha=SMOOTHING_ALPHA):
    g = train_slice.groupby(entity_col)["isFraud"].agg(["sum", "count"])
    g["rate"] = (g["sum"] + alpha * global_rate) / (g["count"] + alpha)
    return g["rate"]

nbr_entities = ["card1", "P_emaildomain", "R_emaildomain",
                "addr1", "DeviceInfo", "id_31"]

nbr_features = {}
for e in nbr_entities:
    rate_map = smoothed_fraud_rate(train_df, e)
    nbr_features[f"{e}__nbr_fraud_rate"] = df[e].map(rate_map).fillna(global_rate)

nbr_df = pd.DataFrame(nbr_features)
print("Neighbor fraud-rate features created:", list(nbr_df.columns))
nbr_df.describe().round(4)

Neighbor fraud-rate features created: ['card1__nbr_fraud_rate', 'P_emaildomain__nbr_fraud_rate', 'R_emaildomain__nbr_fraud_rate', 'addr1__nbr_fraud_rate', 'DeviceInfo__nbr_fraud_rate', 'id_31__nbr_fraud_rate']


,card1__nbr_fraud_rate,P_emaildomain__nbr_fraud_rate,R_emaildomain__nbr_fraud_rate,addr1__nbr_fraud_rate,DeviceInfo__nbr_fraud_rate,id_31__nbr_fraud_rate
count,590540.0000,590540.0000,590540.0000,590540.0000,590540.0000,590540.0000
mean,0.0354,0.0359,0.0454,0.0260,0.0410,0.0447
std,0.0514,0.0143,0.0274,0.0123,0.0372,0.0295
min,0.0003,0.0015,0.0006,0.0033,0.0014,0.0024
25%,0.0099,0.0229,0.0351,0.0200,0.0351,0.0351
50%,0.0199,0.0351,0.0351,0.0245,0.0351,0.0351
75%,0.0389,0.0440,0.0351,0.0307,0.0351,0.0351
max,0.8132,0.4072,0.7670,0.3662,0.9018,0.5870


## 4. Amount aggregates

In [5]:
amt_features = {}
for e in ["uid", "card1", "P_emaildomain", "addr1"]:
    mean = df.groupby(e)["TransactionAmt"].transform("mean")
    std  = df.groupby(e)["TransactionAmt"].transform("std")
    amt_features[f"{e}__amt_mean"]  = mean
    amt_features[f"{e}__amt_std"]   = std
    amt_features[f"{e}__amt_ratio"] = df["TransactionAmt"] / (mean + 1e-6)

amt_df = pd.DataFrame(amt_features)
print("Amount aggregate features created:", list(amt_df.columns))

Amount aggregate features created: ['uid__amt_mean', 'uid__amt_std', 'uid__amt_ratio', 'card1__amt_mean', 'card1__amt_std', 'card1__amt_ratio', 'P_emaildomain__amt_mean', 'P_emaildomain__amt_std', 'P_emaildomain__amt_ratio', 'addr1__amt_mean', 'addr1__amt_std', 'addr1__amt_ratio']


## 5. Velocity features

In [ ]:
def rolling_txn_count(df, entity_col, window_sec):
    """Count of prior transactions on this entity within `window_sec` seconds."""
    tmp = df[[entity_col, "TransactionDT"]].copy()
    tmp["one"] = 1
    tmp = tmp.sort_values([entity_col, "TransactionDT"])
   
    out = np.zeros(len(tmp), dtype=np.int32)
    for _, grp in tmp.groupby(entity_col, sort=False):
        times = grp["TransactionDT"].to_numpy()
        left = 0
        for right in range(len(times)):
            while times[right] - times[left] > window_sec:
                left += 1
            out[grp.index[right]] = right - left  
    tmp["count"] = out
    return tmp.sort_index()["count"]

vel_features = {}
for e in ["uid", "card1"]:
    for window_label, window_sec in [("1h", 3600), ("24h", 86400), ("7d", 7 * 86400)]:
        key = f"{e}__vel_{window_label}"
        vel_features[key] = rolling_txn_count(df, e, window_sec)
        print(f"  computed {key}")

vel_df = pd.DataFrame(vel_features)
vel_df.describe().round(2)

  computed uid__vel_1h
  computed uid__vel_24h
  computed uid__vel_7d
  computed card1__vel_1h
  computed card1__vel_24h
  computed card1__vel_7d


,uid__vel_1h,uid__vel_24h,uid__vel_7d,card1__vel_1h,card1__vel_24h,card1__vel_7d
count,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00
mean,0.37,1.36,2.11,1.46,18.80,108.63
std,4.59,21.07,27.63,5.50,45.61,187.34
min,0.00,0.00,0.00,0.00,0.00,0.00
25%,0.00,0.00,0.00,0.00,1.00,5.00
50%,0.00,0.00,0.00,0.00,5.00,34.00
75%,0.00,0.00,1.00,1.00,19.00,126.00
max,191.00,880.00,975.00,191.00,880.00,1678.00


## 6. Assemble & save

In [7]:
graph = pd.concat([deg_df, nbr_df, amt_df, vel_df], axis=1)
graph.insert(0, "TransactionID", df["TransactionID"].values)

print(f"Graph feature matrix: {graph.shape}")
print(f"Columns: {list(graph.columns)}")

graph.to_parquet(DATA_OUT, index=False)
print(f"Saved {DATA_OUT}")
print("Next step: 05_combined_model.ipynb")

Graph feature matrix: (590540, 37)
Columns: ['TransactionID', 'uid__nunique_card1', 'uid__nunique_P_emaildomain', 'uid__nunique_DeviceInfo', 'uid__nunique_addr1', 'card1__nunique_uid', 'card1__nunique_P_emaildomain', 'card1__nunique_addr1', 'P_emaildomain__nunique_uid', 'P_emaildomain__nunique_card1', 'addr1__nunique_uid', 'addr1__nunique_card1', 'DeviceInfo__nunique_uid', 'card1__nbr_fraud_rate', 'P_emaildomain__nbr_fraud_rate', 'R_emaildomain__nbr_fraud_rate', 'addr1__nbr_fraud_rate', 'DeviceInfo__nbr_fraud_rate', 'id_31__nbr_fraud_rate', 'uid__amt_mean', 'uid__amt_std', 'uid__amt_ratio', 'card1__amt_mean', 'card1__amt_std', 'card1__amt_ratio', 'P_emaildomain__amt_mean', 'P_emaildomain__amt_std', 'P_emaildomain__amt_ratio', 'addr1__amt_mean', 'addr1__amt_std', 'addr1__amt_ratio', 'uid__vel_1h', 'uid__vel_24h', 'uid__vel_7d', 'card1__vel_1h', 'card1__vel_24h', 'card1__vel_7d']
Saved /Users/shengqu/Desktop/CSCI5253/FinalProject/ieee-fraud-detection/data/graph_features.parquet
Next step